# langstage-core in Jupyter — streaming a LangGraph agent with `iter_chunk_frames`

`langstage_core.agui.iter_chunk_frames` is the Jupyter/CLI mapping: it drives any
`CompiledGraph` in-process and yields small `stream_graph_updates` chunk dicts you
render however you like. The frame vocabulary is:

- `{"status": "streaming", "chunk": "...", "node": "..."}` — a token delta of the answer
- `{"status": "streaming", "reasoning": "..."}` — a reasoning-model chain-of-thought delta
- `{"status": "streaming", "tool_calls": [...]}` / `{"status": "streaming", "tool_result": {...}}`
- `{"status": "interrupt", ...}` — a human-in-the-loop approval request
- `{"status": "complete"}` / `{"status": "error", ...}`

> Requirements: `pip install "langstage-core[agui]" langgraph`. The example agent is
> keyless (a one-node echo graph), so this notebook runs with no API key. Swap in a
> model-backed agent (see the last cell) to see `tool_calls` / `interrupt` frames.

## Setup

`agent.py` (next to this notebook) is the same keyless echo agent the FastAPI example
uses — any `CompiledGraph` works here.

In [ ]:
from langstage_core.agui import build_agent, iter_chunk_frames
from agent import agent as graph

# iter_chunk_frames drives an already-built AG-UI LangGraphAgent; build_agent wraps
# any CompiledGraph into one.
agent = build_agent(graph, name="Example")

## The raw frames

Each frame is a small dict — no nested LangGraph state to wade through. (Jupyter runs
an event loop, so top-level `async for` works in a cell.)

In [ ]:
async for frame in iter_chunk_frames(agent, "hello from Jupyter", thread_id="demo"):
    print(frame)

## Accumulate the answer live

Append `chunk` deltas to render the streamed reply in place (`IPython.display` clears
and re-draws the cell output as tokens arrive).

In [ ]:
from IPython.display import clear_output

answer = ""
async for frame in iter_chunk_frames(agent, "stream this back to me", thread_id="demo-2"):
    if frame.get("chunk"):
        answer += frame["chunk"]
        clear_output(wait=True)
        print(answer)

## Tools, interrupts, and resuming

A model-backed agent with tools also emits `tool_calls` / `tool_result` frames, and — if
it uses a human-in-the-loop middleware — a `{"status": "interrupt", ...}` frame. Answer it
by calling `iter_chunk_frames` again on the **same `thread_id`** with `resume=` (the value
rides `forwarded_props.command.resume` → LangGraph `Command(resume=...)`). The resume value
is the **decision envelope** the HITL middleware reads back — a dict with a `"decisions"`
list (the same shape `create_resume_input(decisions=[...])` builds):

```python
# after an {"status": "interrupt", ...} frame on thread "demo":
async for frame in iter_chunk_frames(
    agent, "", thread_id="demo", resume={"decisions": [{"type": "approve"}]}
):
    print(frame)
```

To try it for real, swap the keyless agent for a model-backed one and re-run the cells
above (needs an API key):

```python
# pip install deepagents langchain-anthropic   # + export ANTHROPIC_API_KEY
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver

graph = create_deep_agent(name="Example", checkpointer=InMemorySaver())
agent = build_agent(graph, name="Example")
```